In [1]:
# Instalasi library ANN (jika dijalankan di Google Colab / lingkungan baru)
# !pip install annoy faiss-cpu hnswlib

In [2]:
import pandas as pd
import numpy as np
import time
import os
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Load dataset
# -------------------------------
# Path otomatis mendeteksi lokasi file di lingkungan lokal (VS Code) maupun Google Colab
if os.path.exists('spotify_songs.csv'):
    dataset_path = 'spotify_songs.csv'
elif os.path.exists('Jobsheet6/spotify_songs.csv'):
    dataset_path = 'Jobsheet6/spotify_songs.csv'
else:
    dataset_path = '/content/drive/MyDrive/Kuliah/2025 ML/spotify_songs.csv'

df = pd.read_csv(dataset_path)

features = ['danceability', 'energy', 'loudness', 'speechiness', 
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors

print(f"Dataset shape: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"Fitur numerik ({len(features)}): {features}")
print(f"Bentuk matriks fitur terstandarisasi (X_scaled): {X_scaled.shape}")

Dataset shape: 32833 baris, 23 kolom
Fitur numerik (9): ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
Bentuk matriks fitur terstandarisasi (X_scaled): (32833, 9)


In [3]:
# -------------------------------
# Exact Nearest Neighbor (brute-force)
# -------------------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

Exact NN done in 0.385 s


In [4]:
# -------------------------------
# Annoy
# -------------------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

Annoy done in 0.574 s


In [5]:
# -------------------------------
# HNSW
# -------------------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
p_hnsw.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, dist_hnsw = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

HNSW done in 0.519 s


In [6]:
# -------------------------------
# FAISS IVF
# -------------------------------
start = time.time()
quantizer = faiss.IndexFlatL2(X_scaled.shape[1])
# Catatan penyesuaian: argumen nlist=100 dan metric diberikan secara positional agar kompatibel dengan C++ SWIG wrapper FAISS
index_faiss = faiss.IndexIVFFlat(quantizer, X_scaled.shape[1], 100, faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
dist_faiss, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

FAISS IVF done in 0.180 s


In [7]:
# -------------------------------
# Contoh tampilkan top-5 neighbors dari item pertama
# -------------------------------
print("\nTop-5 neighbors for first song:")
print(f"Exact NN: {idx_exact[0][:5]}")
print(f"Annoy:    {idx_annoy[0][:5]}")
print(f"HNSW:     {idx_hnsw[0][:5]}")
print(f"FAISS:    {idx_faiss[0][:5]}")


Top-5 neighbors for first song:
Exact NN: [    0 29684 29734  4959 29713]
Annoy:    [0, 29684, 29734, 4959, 302]
HNSW:     [    0 29684 29734  4959 29713]
FAISS:    [    0 29684 29734  4959 29713]


In [8]:
# -------------------------------
# Evaluasi Tambahan: Relevansi Lagu & Recall Keseluruhan
# -------------------------------
target_track = df.iloc[0]
print(f"Target Song (Index 0): '{target_track['track_name']}' by {target_track['track_artist']} [Genre: {target_track['playlist_genre']}]\n")
print("Top-5 Lagu Paling Mirip Berdasarkan Exact NN:")
for rank, idx in enumerate(idx_exact[0][:5], 1):
    row = df.iloc[idx]
    print(f"  {rank}. [Index {idx:5d}] {row['track_name']} - {row['track_artist']} (Genre: {row['playlist_genre']})")

# Perhitungan Recall@k terhadap Exact NN (Ground Truth)
def calc_recall(approx_list, exact_arr, k_val=10):
    total_acc = 0.0
    for approx_row, exact_row in zip(approx_list, exact_arr):
        set_approx = set(approx_row[:k_val])
        set_exact = set(exact_row[:k_val])
        total_acc += len(set_approx.intersection(set_exact)) / k_val
    return (total_acc / len(approx_list)) * 100

recall_annoy = calc_recall(idx_annoy, idx_exact, k)
recall_hnsw = calc_recall(idx_hnsw, idx_exact, k)
recall_faiss = calc_recall(idx_faiss, idx_exact, k)

# Ringkasan Perbandingan dalam Bentuk DataFrame
summary_table = pd.DataFrame({
    'Metode': ['Exact NN (Brute-force)', 'Annoy', 'HNSW (hnswlib)', 'FAISS IVF'],
    'Struktur / Algoritma': [
        'Linear Search (k-NN)', 
        'Random Projection Trees (10 trees)', 
        'Hierarchical Graph (M=16, ef=200)', 
        'Inverted File Index (nlist=100, nprobe=10)'
    ],
    'Waktu Total (s)': [time_exact, time_annoy, time_hnsw, time_faiss],
    f'Recall@{k} (%)': [100.0, recall_annoy, recall_hnsw, recall_faiss]
})

print("\nTabel Perbandingan Performa Algoritma ANN:")
display(summary_table)

Target Song (Index 0): 'I Don't Care (with Justin Bieber) - Loud Luxury Remix' by Ed Sheeran [Genre: pop]

Top-5 Lagu Paling Mirip Berdasarkan Exact NN:
  1. [Index     0] I Don't Care (with Justin Bieber) - Loud Luxury Remix - Ed Sheeran (Genre: pop)
  2. [Index 29684] I Don't Care (with Justin Bieber) - Loud Luxury Remix - Ed Sheeran (Genre: edm)
  3. [Index 29734] I Miss You (feat. Julia Michaels) - Cahill Remix - Clean Bandit (Genre: edm)
  4. [Index  4959] Name & Number - Shift K3Y (Genre: pop)
  5. [Index 29713] Rise - TV Noise Ibiza Mix - Jonas Blue (Genre: edm)



Tabel Perbandingan Performa Algoritma ANN:


,Metode,Struktur / Algoritma,Waktu Total (s),Recall@10 (%)
0,Exact NN (Brute-force),Linear Search (k-NN),0.385232,100.000000
1,Annoy,Random Projection Trees (10 trees),0.574401,87.684037
2,HNSW (hnswlib),"Hierarchical Graph (M=16, ef=200)",0.518853,98.795419
3,FAISS IVF,"Inverted File Index (nlist=100, nprobe=10)",0.179638,99.240398
